In [4]:
# tf_nn.py
import time
import tensorflow as tf
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

tf.random.set_seed(0)
print("TensorFlow version:", tf.__version__)

# Boston regression
boston = fetch_openml(name="Boston", version=1, as_frame=False)
Xb = StandardScaler().fit_transform(boston["data"])
yb = MinMaxScaler((0, 1)).fit_transform(boston["target"].reshape(-1, 1))

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(Xb.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])
model.compile(optimizer=tf.keras.optimizers.SGD(0.01), loss="mse")

epochs = 200
t0 = time.perf_counter()
model.fit(Xb, yb, epochs=epochs, verbose=0)
loss_val = model.evaluate(Xb, yb, verbose=0)
print(f"[Boston][TF] MSE={loss_val:.6f} elapsed={time.perf_counter()-t0:0.1f}s\n")

# MNIST multiclass (full)
mn = fetch_openml("mnist_784", version=1, as_frame=False)
X = mn["data"].astype(np.float32) / 255.0
y = mn["target"].astype(int)
X = StandardScaler().fit_transform(X)
enc = OneHotEncoder(sparse_output=False)
yoh = enc.fit_transform(y.reshape(-1, 1))

model2 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X.shape[1],)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])
model2.compile(optimizer=tf.keras.optimizers.SGD(0.1), loss="categorical_crossentropy", metrics=["accuracy"])

epochs = 5
t0 = time.perf_counter()
history = model2.fit(X, yoh, epochs=epochs, verbose=0)
loss, acc = model2.evaluate(X, yoh, verbose=0)
print(f"[MNIST][TF] loss={loss:.4f} acc={acc:.4f} elapsed={time.perf_counter()-t0:0.1f}s\n")

# Pima binary
pima = fetch_openml("pima-indians-diabetes", version=1, as_frame=False)
Xp = StandardScaler().fit_transform(pima["data"])
yp = pima["target"].astype(float).reshape(-1, 1)

model3 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(Xp.shape[1],)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])
model3.compile(optimizer=tf.keras.optimizers.SGD(0.01), loss="binary_crossentropy", metrics=["accuracy"])
epochs = 100
t0 = time.perf_counter()
model3.fit(Xp, yp, epochs=epochs, verbose=0)
loss3, acc3 = model3.evaluate(Xp, yp, verbose=0)
print(f"[Pima][TF] BCE={loss3:.6f} acc={acc3:.4f} elapsed={time.perf_counter()-t0:0.1f}s")


TensorFlow version: 2.20.0
[Boston][TF] MSE=0.008738 elapsed=8.3s

[MNIST][TF] loss=0.0462 acc=0.9891 elapsed=12.6s

[Pima][TF] BCE=0.442091 acc=0.7917 elapsed=6.3s
